In [1]:
import ipywidgets as widgets
import pyarrow as pa
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import base64
from io import BytesIO
from PIL import Image
import json

In [2]:
path = "viewfs://hadoop-lt-cluster/home/reco_wl/mpi/luoxinchen/recovlm_dataset_l2/p_date=20250113"
fs = pa.hdfs.connect(user='mpi')
files = fs.ls(path)
files = [x for x in files if x.endswith("parquet")]

In [3]:
x = np.random.choice(files)
print(x)
df = pq.read_table(x).to_pandas()

viewfs://hadoop-lt-cluster/home/reco_wl/mpi/luoxinchen/recovlm_dataset_l2/p_date=20250113/rank-232-afa2fe9c-d31b-11ef-aec0-946daee91052.parquet


In [10]:
def decode_base64_image(base64_string):
    # 解码base64字符串并返回PIL图像对象
    image_data = base64.b64decode(base64_string)
    image = Image.open(BytesIO(image_data))
    return image

def format_messages(messages, images):
    for msg in messages:
        msg_content = f"[{msg['role']}]: "
        for content in msg["content"]:
            if content["type"] == "image":
                if msg_content != "":
                    print(msg_content)
                    msg_content = ""
                name = content["image"]
                image = images[name]
                display(name, image.size, image)
            elif content["type"] == "text":
                msg_content += content["text"]
            elif content["type"] == "video":
                print(content)
        if msg_content != "":
            print(msg_content)
        print("-" * 10)

def format_segments(segments, images):
    for seg in segments:
        if seg["type"] == "text":
            print(seg["text"] + "\n")
        elif seg["type"] == "image":
            name = seg["image"]
            image = images[name]
            display(name, image.size, image)
            

def display_sample(sample):
    source = sample['source']
    images = json.loads(sample.pop('images'))
    messages = json.loads(sample['messages'])
    segments = json.loads(sample['segments'])
    for name, base64_string in images.items():
        image = decode_base64_image(base64_string)
        images[name] = image

    print(f"source={source}, img_nums={len(images)}")
    if segments is not None:
        print("segments sample. Content:")
        print("=" * 10)
        format_segments(segments, images)
    elif messages is not None:
        print("message sample. Content:")
        print("=" * 10)
        format_messages(messages, images)

display_sample(df.sample().iloc[0].to_dict())

source=MMC4-ff, img_nums=0
segments sample. Content:
Last week we marked the 10-year anniversary of the financial crisis.

Today we’ll look at some of the lessons traders can take from the event for their own use in the future.

The first lesson is to recognize and respect price trends.

The S&P 500, after all, made successively lower lows in March, July and September of 2008.

Lower highs followed in April and August.

This is the textbook definition of a bearish trend.

The second lesson is that trends often accelerate before they finish.

The drops earlier in 2008 were pip squeaks compared with the massacre that would come in October: Six percent in January… 9 percent in June and September, followed by a 17 percent crash in October.

Something to think about the next time you’re trying to bottom-fish a bearish stock or sell short a rallying one.

About two weeks after this exchange, Lehman was bankrupt.

Obviously not all cases are this obvious, but it illustrates how even highly ed